In [ ]:
%run ./_utils

### Export `merged_ids.csv.gz` for authors and sources (oxjob #1124)

Writes the loser -> winner maps built nightly by `notebooks/elastic/sync_merged_ids`
(`openalex.authors.merged_author_ids`, `openalex.sources.merged_source_ids`) to
`full/{date}/{format}/{authors,sources}/merged_ids.csv.gz` in both format trees, next to each
entity's `manifest.json`, the way `export_deleted_works` writes `works/deleted_ids.csv.gz`.
Columns: `id` (the merged-away ID), `merge_into_id` (the live record it now resolves to, chains
already followed), `merge_date`; IDs in the same full-URL form as the entity files. The file is
cumulative and is the whole truth each day: a loser that comes back to life drops out of it.
Header-only when the map table does not exist yet.

Task ordering: a leaf after `update_meta` (so after `export_authors` and `export_sources`, which
`rm -r` their entity directories first). A failure here never blocks the manifests, smoke tests or
the public sync. It removes its own `_temp/merged_ids_*` scratch.

In [ ]:
MAPS = {
    "authors": ("openalex.authors.merged_author_ids", "https://openalex.org/A"),
    "sources": ("openalex.sources.merged_source_ids", "https://openalex.org/S"),
}

date_str = get_snapshot_date()


def single_part(path, suffix):
    parts = [f.path for f in dbutils.fs.ls(path)
             if f.name.startswith("part-") and f.name.endswith(suffix)]
    if len(parts) != 1:
        raise RuntimeError(f"Expected one merged-ids part ending in {suffix}, found {len(parts)}")
    return parts[0]


for entity, (table, url) in MAPS.items():
    if spark.catalog.tableExists(table):
        df = (spark.table(table)
              .selectExpr(f"CONCAT('{url}', id) AS id", f"CONCAT('{url}', merge_into_id) AS merge_into_id",
                          "merge_date")
              .coalesce(1)
              .sortWithinPartitions("merge_date", "id"))
        n = spark.table(table).count()
    else:
        df = spark.createDataFrame([], "id STRING, merge_into_id STRING, merge_date DATE").coalesce(1)
        n = 0
        print(f"{table} does not exist yet; writing a header-only file")

    tmp_dir = f"{S3_BASE}/{date_str}/_temp/merged_ids_{entity}"
    (df.write.mode("overwrite").option("header", True).option("compression", "gzip").csv(tmp_dir))
    part = single_part(tmp_dir, ".csv.gz")
    for fmt in ("jsonl", "parquet"):
        out_path = f"{S3_BASE}/{date_str}/{fmt}/{entity}/merged_ids.csv.gz"
        if not dbutils.fs.cp(part, out_path):
            raise RuntimeError(f"Failed to copy merged-ids map to {out_path}")
        print(f"wrote {n:,} merged {entity} to {out_path}")
    dbutils.fs.rm(tmp_dir, recurse=True)